# Pulse Sequencer Configuration using SCPI

## Overview
This document covers the basics of configuring a pulse sequencer using the SCPI interface of an OpenSync device. 

## Purpose
This example demonstrates how to communicate an OpenSync device using its SCPI interface and configure a pulse sequencer using the raw commands. It is assumed that you have some fundamental knowledge of how SCPI commands work. If not, then one can read the basics [here](). Additionally, OpenSync relies on a stable USB connection, so make sure one is present for this demonstration.

## Imports

In [1]:
from opensync import opensync

## Device Connection

In [2]:
avail_ports = opensync.device_comm_search()
avail_ports

['COM7']

## Device Information

In [3]:
with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    print(opensync.device_system_version(sync_device))

['OpenPIV', 'OpenSync', '5A3F6CE6128EEB36', '0.7.3']


## Configuring a Pulse Sequencer
A pulse sequencer on an OpenSync device outputs a pulse sequencer through eight (8) output channels. It is only concerned with producing an output sequence once an internal trigger is recieved and not where that signal came from or its repetition frequency. A pulse sequencer synthesizes a pulse sequence through output states and delays to the next output state change. An output state is merely the bit pattern of high and low pins (e.g., the output channels) at a given time. The delay between output states is measured in seconds and later converted into clock cycles.

Now, lets say that we want to output a pulse sequence where channel 1 (index 0) is high for 2 seconds, channel 2 (index 1) is high for 2 seconds 1 second after channel 1 turn high, and channel 2 (index 2) turns high when channels 1 and 2 go low. The following pulse sequence will look like the following.
![Example Pulse Sequence](images/example_pulse_timing.png)

### Load instructions to device

Each channel consists of a dedicated sequencer and can be independently programmed. This simplifies programming of the device since simple output states and delays can be used compared to earlier firmware versions of OpenSync. For instancem channel instructions simply consist of output state and delay pairs which are intuitive to visualize.

Since we are working with the direct SCPI interface and not the Python wrapper API, we will be setting the units of the delays directly in the instructions. Note, you do not have to manually specify the end of a pulse as it is automaticall assumed that the output state will be off after a pulse instruction.

In [4]:
commands_send = [
    'pulse1:buffer on, 2 s',
    'pulse2:buffer off, 1 s, on, 2 s',
    'pulse3:buffer off, 3 s, on, 2 s'
]

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    for command in commands_send:
        # Write output states to cache
        resp = opensync.device_comm_write(
            sync_device,
            command
        )
    
        # Note, a blank response means a success.
        print(resp)

[]
[]
[]


Let's say that the pulse on channel 3 is to be 10 seconds long instead. If a delay is too long for the current clock resolution, the device will abort loading the parameters.

In [5]:
command = 'pulse3:buffer off, 3 s, on, 10 s'

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    # Write output states to cache
    resp = opensync.device_comm_write(
        sync_device,
        command
    )

    # Note, a blank response means a success.
    print(resp)

['**ERROR: -222', ' "Data out of range"']


If we have delays of an unsupported length, we can change the clock divider so the clock resolution is lower. Here, we chose 2 which results in an 8 ns clock resolution. You can also simply split the delay between two instruction pairs which would accomplish the same thing.

In [6]:
command = 'pulse3:divider 2'

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    # Change data units to seconds for pulse sequencer 0
    resp = opensync.device_comm_write(
        sync_device,
        command
    )
    
    print(resp)

[]


With the updated clock divider, we can now re-apply the new pulse instructions to sequencer 0.

In [7]:
command = 'pulse3:buffer off, 3 s, on, 10 s'

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    # Write output states to cache
    resp = opensync.device_comm_write(
        sync_device,
        command
    )
    
    # Note, a blank response means a success.
    print(resp)

[]


### Activating the pulse sequencer

Now that the pulse sequences have been loaded to the device, we can enable the pulse sequencer and select a sync source which triggers the pulse sequence to execute. Sync sources may be other channels as long as the primary channel is synced to `T0`. This is not currently enforced in the firmware, so beware when syncing to other channels to make sure there are no circular syncing dependencies.

In [8]:
commands = [
    'pulse1:sync t0',
    'pulse2:sync t0',
    'pulse3:sync t0'
]

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    for command in commands_send:
        # Write output states to cache
        resp = opensync.device_comm_write(
            sync_device,
            command
        )
    
        # Note, a blank response means a success.
        print(resp)

[]
[]
[]


In [9]:
commands = [
    'pulse1:state on',
    'pulse2:state on',
    'pulse3:state on'
]

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    for command in commands_send:
        # Write output states to cache
        resp = opensync.device_comm_write(
            sync_device,
            command
        )
    
        # Note, a blank response means a success.
        print(resp)

[]
[]
[]


The pulse sequence will now be executed every time the clock sequencer sends a signal during program operation.

## Configuring a Clock Sequencer
The clock sequencer operates in a very similar manner to the pulse sequencers. While we only touched the very basics of the pulse sequencer, many of the clock sequencer functions discussed below are also available in the pulse sequencers. The purpose of the clock sequencer is to provide a reference time of which the pulse sequencers will execute. The clock sequencer consists of an internal frequency function and a trigger/gate function. The essence of the internal frequency is what will be discussed below.

The frequency function can be programmed using the following commands.
 - `state`: Enables a clock sequencer. If disabled, the pulse sequencers will never start.
 - `divider`: Sets the clock divider of the sequencer.
 - `period`: Sets the period of the frequency.
 - `bcounter`: Sets the number of trigger events to accept before stopping the program.
 - `pcounter`: Sets the active portion of the duty cycle. It basically acts as normal, single shot, or burst mode in commercial synchronizing devices.
 - `ocounter`: Sets the inactive portion of the duty cycle. This is how many internal clock periods to skip before producing p counter pulses again.

For our pulse sequence we programmed above, it has a period of around 5 seconds, so we will add an extra second to allow for some time between pulses. With the addition of the extra 10 second delay, the total period we will use would be something along the lines of 14 seconds. We would like to have 3 pulse sets per trigger, so we will set p counter to 3.

In [10]:
command = 'pulse0:state on'

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    # Write output states to cache
    resp = opensync.device_comm_write(
        sync_device,
        command
    )
    
    # Note, a blank response means a success.
    print(resp)

[]


In [11]:
command = 'pulse0:period 14 s'

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    # Write output states to cache
    resp = opensync.device_comm_write(
        sync_device,
        command
    )
    
    # Note, a blank response means a success.
    print(resp)

[]


In [12]:
command = 'pulse0:bcounter 3'

with opensync.device_comm_managed(avail_ports[0]) as sync_device:
    # Write output states to cache
    resp = opensync.device_comm_write(
        sync_device,
        command
    )
    
    # Note, a blank response means a success.
    print(resp)

[]


Now the device is ready to operate. Either push the run/stop button or fire the command `device:start` and a pulse sequence will begin.